# Process Compliance & WorkflowConverted from `src/process_compliance.py`---

**Beschreibung:** Process Compliance - Checks adherence to workflow process

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Expected ideal linear process
EXPECTED_PROCESS = ["Open", "In Progress", "Resolved", "Closed"]

# Allowed status transitions (defines what is legally possible)
VALID_TRANSITIONS = {
    "Open":       ["In Progress", "Closed"],
    "In Progress": ["Waiting", "Resolved", "Open"],
    "Waiting":    ["In Progress", "Resolved"],
    "Resolved":   ["Closed", "In Progress"],
    "Closed":     ["In Progress"]   # usually reopen
}


def check_compliance(status_history):
    """
    Check whether a single ticket's status history follows allowed transitions
    and doesn't contain unnecessary backward movements in the main flow.

    Args:
        status_history: list of status strings (chronological)

    Returns:
        dict with compliance metrics
    """
    if not status_history or len(status_history) < 2:
        return {
            'is_compliant': True,
            'violations': 0,
            'backward_steps': 0,
            'compliance_score': 1.0
        }

    violations = 0
    backward_steps = 0

    for i in range(1, len(status_history)):
        prev = status_history[i-1]
        curr = status_history[i]

        # 1. Check allowed transition
        allowed_next = VALID_TRANSITIONS.get(prev, [])
        if curr not in allowed_next:
            violations += 1

        # 2. Detect backward movement in core process
        if prev in EXPECTED_PROCESS and curr in EXPECTED_PROCESS:
            prev_idx = EXPECTED_PROCESS.index(prev)
            curr_idx = EXPECTED_PROCESS.index(curr)
            if curr_idx < prev_idx:
                backward_steps += 1

    total_transitions = len(status_history) - 1
    compliance_score = 1.0 - (violations / total_transitions) if total_transitions > 0 else 1.0

    return {
        'is_compliant': violations == 0,
        'violations': violations,
        'backward_steps': backward_steps,
        'compliance_score': max(0.0, round(compliance_score, 3))
    }


def analyze_workflow_from_wfe(issues_df):
    """
    Analyze workflow compliance using wfe_* columns (workflow execution counts per status).

    Args:
        issues_df: DataFrame containing at least 'id' and wfe_* columns

    Returns:
        DataFrame with one row per issue + compliance metrics
    """
    print("Analyzing workflows...")

    wfe_cols = [col for col in issues_df.columns if col.startswith('wfe_')]
    if not wfe_cols:
        print("Warning: No columns starting with 'wfe_' found.")

    results = []
    total_issues = len(issues_df)

    for idx, row in issues_df.iterrows():
        issue_id = row.get('id', row.get('key', idx))

        total_steps = sum(row[col] for col in wfe_cols if pd.notna(row[col]))

        reopens = row.get('wfe_reopened', 0) if 'wfe_reopened' in issues_df.columns else 0
        reopens = int(reopens) if pd.notna(reopens) else 0

        backward = 0
        for col in wfe_cols:
            count = row[col]
            if pd.notna(count) and count > 1:
                backward += (count - 1)

        penalty = (reopens * 0.10) + (backward * 0.05)
        compliance_score = max(0.0, 1.0 - penalty)

        results.append({
            'issue_id':         issue_id,
            'total_steps':      int(total_steps),
            'reopens':          reopens,
            'backward_steps':   backward,
            'compliance_score': round(compliance_score, 3),
            'is_compliant':     compliance_score > 0.80
        })

        if (idx + 1) % 10000 == 0:
            print(f"   {idx+1:,} / {total_issues:,} processed...")

    print(f" {len(results):,} issues analyzed")
    return pd.DataFrame(results)


def get_compliance_summary(workflow_df):
    """
    Create a high-level summary of workflow compliance across all issues.
    """
    if workflow_df.empty:
        return {"error": "No data to summarize"}

    return {
        'total_issues':           len(workflow_df),
        'compliant_count':        int(workflow_df['is_compliant'].sum()),
        'compliance_rate_%':      round(workflow_df['is_compliant'].mean() * 100, 1),
        'avg_compliance_score':   round(workflow_df['compliance_score'].mean(), 3),
        'avg_steps_per_issue':    round(workflow_df['total_steps'].mean(), 1),
        'total_reopens':          int(workflow_df['reopens'].sum()),
        'issues_with_reopen_%':   round((workflow_df['reopens'] > 0).mean() * 100, 1),
        'total_backward_steps':   int(workflow_df['backward_steps'].sum())
    }

##  Execution

In [4]:
import pandas as pd
from pathlib import Path

print("=" * 50)
print(" PROZESS-COMPLIANCE")
print("=" * 50)

# ─── Load issues data ───
data_path = Path("data/raw/issues.csv")

if data_path.exists():
    issues = pd.read_csv(data_path)
    print(f" Loaded: {len(issues):,} Issues")
    
    # Run workflow compliance analysis
    workflow_df = analyze_workflow_from_wfe(issues)
    
    # Get summary statistics
    summary = get_compliance_summary(workflow_df)
    
    # Print nice summary
    print("\nZUSAMMENFASSUNG:")
    print(f"   Total Issues:          {summary['total_issues']:,}")
    print(f"   Compliant:             {summary['compliant_count']:,} "
          f"({summary['compliance_rate_%']:.1f}%)")
    print(f"   Ø Compliance Score:    {summary['avg_compliance_score']:.3f}")
    print(f"   Reopen Rate:           {summary['issues_with_reopen_%']:.1f}%")
    print(f"   Total Reopens:         {summary['total_reopens']:,}")
    print(f"   Average Steps/Issue:   {summary['avg_steps_per_issue']:.1f}")
    
    # Export results
    output_path = Path("data/processed/workflow_analysis.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    workflow_df.to_csv(output_path, index=False)
    print(f"\n Saved: {output_path}")
    
else:
    print("Issues-Datei nicht gefunden!")
    print(f"  Gesuchter Pfad: {data_path.absolute()}")
    print("  → Prüfe:")
    print("    • Dateiname und Ordner (data/raw/)")
    print("    • Aktuelles Arbeitsverzeichnis:")
    print(f"      {Path('.').absolute()}")

 PROZESS-COMPLIANCE
Issues-Datei nicht gefunden!
  Gesuchter Pfad: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues.csv
  → Prüfe:
    • Dateiname und Ordner (data/raw/)
    • Aktuelles Arbeitsverzeichnis:
      /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks
